# POLITE — capture palette

Supervised acquisition cells. Use only after the device configuration and safety read-backs have been checked. Do not run these while `execute_night.py --run` owns the camera.


## Preamble


In [ ]:
import os, sys
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root: _root = _root.parent
if not (_root / 'pyproject.toml').exists(): raise RuntimeError('Could not locate the POLITE repository root')
os.chdir(_root)
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
print('POLITE root:', _root)


In [ ]:
from pathlib import Path
from obs_utils import live
from obs_utils import interactive as obs
from obs_utils import user_config as uc
SESSION_DIR = None  # set in Tonight's card


## Session and one-frame checks


In [ ]:
s = obs.session()
if SESSION_DIR is None: raise RuntimeError("Set SESSION_DIR in Tonight's card")


In [ ]:
# MOTION — exposure only; returns an array and writes no FITS.
image = s.expose(0.1)
print(image.shape, image.min(), image.max())


In [ ]:
# MOTION — writes one supervised FITS.
path = s.expose(0.1, out_path=SESSION_DIR / 'probe.fits')
live.frame_report(path)


## Bounded live preview

This is deliberately a short `for` loop, not a continuous acquisition loop. If a cell is interrupted during an exposure, wait for the camera to finish or run the recovery cell below before starting another frame.


In [ ]:
# MOTION — bounded, RAM-only live preview; do not replace this with while True.
from IPython.display import clear_output
import matplotlib.pyplot as plt
PREVIEW_N, PREVIEW_EXP_S = 3, 0.01
for _ in range(PREVIEW_N):
    image = s.expose(PREVIEW_EXP_S, timeout_s=30.0)
    clear_output(wait=True)
    plt.imshow(image, origin='upper', cmap='gray'); plt.show()


## Focus run — human chooses the minimum


In [ ]:
positions = [float(v) for v in input('Focuser positions, comma-separated: ').split(',')]
exposure_s = float(input('Unsaturated focus exposure (s): '))


In [ ]:
# MOTION — moves the focuser and takes one frame per position.
sweep = s.focus_sweep(positions, exposure_s)
live.focus_curve(sweep)


## Centre a characterized Savart pair


In [ ]:
# MOTION only when apply=True and detector-to-sky orientation is measured.
result = s.center_on_pair(exposure_s=0.1, apply=False)
result


## Biases and darks — the QHY has no shutter


In [ ]:
BIAS_N = 20
DARK_EXP_S, DARK_N = 10.0, 10


In [ ]:
# MOTION — opaque Dark slot is mandatory.
from alpyca_tools.fits_writer import FitsHeaderConfig
s.filter('Dark')
biases = [s.expose(0.0, out_path=SESSION_DIR / f'bias_{i:03d}.fits', dark=True, header=FitsHeaderConfig(imagetyp='BIAS')) for i in range(BIAS_N)]


In [ ]:
# MOTION — match this to its lights: exposure, gain, mode, binning, and ROI.
from alpyca_tools.fits_writer import FitsHeaderConfig
s.filter('Dark')
darks = [s.expose(DARK_EXP_S, out_path=SESSION_DIR / f'dark_{i:03d}.fits', dark=True, header=FitsHeaderConfig(imagetyp='DARK')) for i in range(DARK_N)]


## Flat probes — adjust exposure for changing twilight


In [ ]:
FLAT_EXP_S = float(input('Unsaturated twilight-flat exposure (s): '))


In [ ]:
# MOTION — one diagnostic flat; use a reviewed night plan for a flat set.
from alpyca_tools.fits_writer import FitsHeaderConfig
flat = s.expose(FLAT_EXP_S, out_path=SESSION_DIR / 'flat_probe.fits', header=FitsHeaderConfig(imagetyp='FLAT'))
live.frame_report(flat)


## Polarimetry — a contiguous four-angle cycle


In [ ]:
OBJECT = input('Target name: ')
HWP_CYCLE = (0.0, 45.0, 22.5, 67.5)
SCIENCE_EXP_S = float(input('Science exposure (s): '))


In [ ]:
# MOTION — restart this complete cycle if either beam clips.
from alpyca_tools.fits_writer import FitsHeaderConfig, PolarimetryCards
cycle = []
for i, angle in enumerate(HWP_CYCLE):
    achieved = s.hwp(angle)
    header = FitsHeaderConfig(object_name=OBJECT, polarimetry=PolarimetryCards(hwp_angle_deg=achieved, pol_seq_id='manual'))
    cycle.append((angle, s.expose(SCIENCE_EXP_S, out_path=SESSION_DIR / f'light_hwp_{angle:g}_{i:02d}.fits', header=header)))
[(angle, live.frame_stats(path).maximum) for angle, path in cycle]


## Polarimetric flats and block ledger


In [ ]:
# MOTION — one flat at each HWP angle; use only when the reviewed plan calls for it.
from alpyca_tools.fits_writer import FitsHeaderConfig, PolarimetryCards
for angle in HWP_CYCLE:
    achieved = s.hwp(angle)
    header = FitsHeaderConfig(imagetyp='FLAT', polarimetry=PolarimetryCards(hwp_angle_deg=achieved, pol_seq_id='flat'))
    s.expose(FLAT_EXP_S, out_path=SESSION_DIR / f'polflat_hwp_{angle:g}.fits', header=header)


In [ ]:
from obs_utils.block_log import BlockLogger
ledger = BlockLogger(SESSION_DIR / 'block_manifest.jsonl')
ledger.append({'block_id': input('Completed block ID: '), 'note': input('Operator note: ')})


## Night plans


In [ ]:
from obs_utils.night_plan import describe, load_night_plan
plan = load_night_plan('night_plans/example.yaml')
print(describe(plan))


Dry-run in a terminal: `python scripts/execute_night.py night_plans/example.yaml`. Add `--run` only after checking that preview and its hardware gates.
